# AMI TS3003 회의전사문 병합 및 전처리

#### TS3003 뒤 a, b, c, d 문자는 하나의 큰 안건을 다루는 회의이지만, 30-40분 단위로 4번에 걸쳐 각각 이루어진 회의임
- 그렇기 때문에 a, b, c, d 각각의 회의로 보아야 하지만, b 회의에서 결정된 안건이 d에서 번복된 경우가 존재하기 때문에 이전 회의에서의 결정사항이 변경된 경우 처리를 하기 좋음
- 또한, a 회의에서 참석자의 간단한 소개를 진행하기 때문에 b, c, d 단계에서도 a 단계의 데이터를 통해 참석자 정보를 어느정도 이해할 수 있음
- 우선은 a, b, c, d 각각의 회의를 하나로 병합했지만, 테스트 시에는 각각의 회의를 다루는 것으로 함

In [ ]:
# 경로 설정
import os
import glob

# 입력 디렉토리
INPUT_DIR = os.path.join("playground", "hyungjin", "test", "input")

# 출력 디렉토리
OUTPUT_DIR = os.path.join("playground", "chaewon", "data")

# 병합 파일 경로
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "TS3003_merged.txt")
MEETING_ID = "TS3003"
HEADER_KEYS = ("회의 대주제", "참여자", "일시")

In [ ]:
# 2. 회의별 전사문 로드 (헤더 / 본문 분리)

def read_transcript(path: str) -> tuple[dict[str, str], str]:
    """전사문 상단의 '키: 값' 헤더와 본문을 분리한다."""

    # 파일 읽기
    with open(path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    header = {}
    i = 0

    while i < len(lines) and ":" in lines[i]:
        key, value = lines[i].split(":", 1)

        if key.strip() not in HEADER_KEYS:
            break

        header[key.strip()] = value.strip()
        i += 1

    body = "\n".join(lines[i:]).strip()

    return header, body


# TS3003으로 시작하는 txt 파일 검색
files = sorted(
    glob.glob(os.path.join(INPUT_DIR, f"{MEETING_ID}*.txt"))
)

# 파일명에서 확장자를 제외한 이름 추출 후 전사문 읽기
transcripts = [
    (
        os.path.splitext(os.path.basename(path))[0],
        *read_transcript(path)
    )
    for path in files
]

# 파일별 헤더 및 본문 길이 확인
for name, header, body in transcripts:
    print(name, header, f"본문 {len(body):,}자", sep="\n  ")

In [ ]:
# 3. 헤더 병합 (동일한 값은 한 번만 표시)
TITLE_RE = re.compile(r"^(?P<project>.+?)\s+(?P<order>\d+차) 회의 - (?P<phase>.+)$")
DATE_RE = re.compile(r"^(?P<date>\S+)\s*\(회의 길이 약 (?P<minutes>\d+)분\)$")


def merge_values(values: list[str]) -> str:
    """값이 모두 같으면 하나만, 다르면 순서대로 이어 붙인다."""
    unique = list(dict.fromkeys(values))
    return unique[0] if len(unique) == 1 else " / ".join(unique)


def parse(pattern: re.Pattern, value: str) -> re.Match:
    match = pattern.match(value)
    if match is None:
        raise ValueError(f"헤더 형식을 해석할 수 없습니다: {value!r}")
    return match


titles = [parse(TITLE_RE, header["회의 대주제"]) for _, header, _ in transcripts]
dates = [parse(DATE_RE, header["일시"]) for _, header, _ in transcripts]
minutes = [int(m["minutes"]) for m in dates]

project = merge_values([m["project"] for m in titles])
sessions = ", ".join(f"{m['order']} {m['phase']}" for m in titles)
participants = merge_values([header["참여자"] for _, header, _ in transcripts])
date = merge_values([m["date"] for m in dates])
durations = ", ".join(f"{t['order']} {d}분" for t, d in zip(titles, minutes))

merged_header = "\n".join([
    f"회의 대주제: {project} {titles[0]['order']}~{titles[-1]['order']} 회의 ({sessions})",
    f"참여자: {participants}",
    f"일시: {date} (회의 길이 약 {sum(minutes)}분: {durations})",
])
print(merged_header)

In [ ]:
# 본문 병합 및 저장
merged_body = "\n\n".join(body for _, _, body in transcripts)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(f"{merged_header}\n\n{merged_body}\n", encoding="utf-8", newline="\n")
print(f"저장 완료: {OUTPUT_PATH} ({len(transcripts)}개 파일 병합)")